In [0]:
%python
dbutils.widgets.text("catalog", "dev")
catalog = dbutils.widgets.get("catalog")

In [0]:
%python
# Sequential notebook runs can be slow because each waits for the previous to finish.
# To speed up, run notebooks in parallel using ThreadPoolExecutor.

from concurrent.futures import ThreadPoolExecutor

notebooks = [
    "customers_type1",
    "products_type2",
    "sales_type1",
    "supplier_type2"
]

def run_notebook(notebook):
    try:
        result = dbutils.notebook.run(notebook, 0, {"catalog": catalog})
        return (notebook, "SUCCESS", result)
    except Exception as e:
        return (notebook, "FAILED", str(e))

with ThreadPoolExecutor() as executor:
    results = list(executor.map(run_notebook, notebooks))

# Check for failures
failures = [r for r in results if r[1] == "FAILED"]
if failures:
    error_details = "\n".join([f"  - {name}: {error}" for name, status, error in failures])
    raise Exception(f"The following notebook(s) failed:\n{error_details}")